# Xpect AI — RAG + LLM Generation

This notebook builds Phase 2 of the Xpect AI RAG system.

## Phase 2 Pipeline

User Question
→ Query Embedding
→ FAISS Retrieval
→ Relevant Movie Context
→ Prompt Construction
→ LLM
→ Natural Language Answer

Phase 1 built the retrieval foundation.

Phase 2 connects the retrieved information to an LLM so that
Xpect AI can generate natural-language answers.

##### Artifacts

In [20]:
import pandas as pd
import numpy as np
import faiss
import pickle
from sentence_transformers import SentenceTransformer
print("Done")

Done


##### Loading the existing vector store

In [21]:
index=faiss.read_index(
    "../vector_store/netflix_index"
)
with open("../vector_store/documents.pkl","rb") as f:
    documents=pickle.load(f)
movies = pd.read_pickle(
    "../vector_store/movies.pkl"
)

print("Vector store loaded successfully!")
print("Movies:", len(documents))
print("Vectors:", index.ntotal)

Vector store loaded successfully!
Movies: 8807
Vectors: 8807


##### Loading Embedding Modek

In [22]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)
print("Embedding model loaded!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1499.85it/s]


Embedding model loaded!


##### Making Retriver

In [23]:
def retrieve_movies(query, top_k=5):
    query_embedding = embedding_model.encode(
        [query]
    ).astype("float32")

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for idx in indices[0]:
        results.append(documents[idx])

    return results

#### Test the query

In [24]:
results = retrieve_movies(
    "movies about a father and daughter"
)
for movie in results:
    print(movie)
    print("-" * 80)


Title : Father of the Year
Type : Movie
Director : Unknown
Cast : David Spade, Nat Faxon, Joey Bragg, Matt Shively, Bridgit Mendler, Jackie Sandler, Mary Gillis
Country : United States
Release Year : 2018
Rating : TV-MA
Duration : 95 min
Genres : Comedies
Description : A drunken debate between two recent college grads about whose father would win in a fight leads to mayhem when their dads take the challenge seriously.

--------------------------------------------------------------------------------

Title : Fatherhood
Type : Movie
Director : Paul Weitz
Cast : Kevin Hart, Alfre Woodard, Lil Rel Howery, DeWanda Wise, Frankie Faison, Anthony Carrigan, Paul Reiser, Melody Hurd
Country : United States
Release Year : 2021
Rating : PG-13
Duration : 111 min
Genres : Dramas
Description : A widowed new dad copes with doubts, fears, heartache and dirty diapers as he sets out to raise his daughter on his own. Inspired by a true story.

-------------------------------------------------------------

### Llm Layer


In [25]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

Client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)
print("API Key loaded")

API Key loaded


### Context Layer

In [26]:
context="\n\n -- \n\n".join(results)
print(context)


Title : Father of the Year
Type : Movie
Director : Unknown
Cast : David Spade, Nat Faxon, Joey Bragg, Matt Shively, Bridgit Mendler, Jackie Sandler, Mary Gillis
Country : United States
Release Year : 2018
Rating : TV-MA
Duration : 95 min
Genres : Comedies
Description : A drunken debate between two recent college grads about whose father would win in a fight leads to mayhem when their dads take the challenge seriously.


 -- 


Title : Fatherhood
Type : Movie
Director : Paul Weitz
Cast : Kevin Hart, Alfre Woodard, Lil Rel Howery, DeWanda Wise, Frankie Faison, Anthony Carrigan, Paul Reiser, Melody Hurd
Country : United States
Release Year : 2021
Rating : PG-13
Duration : 111 min
Genres : Dramas
Description : A widowed new dad copes with doubts, fears, heartache and dirty diapers as he sets out to raise his daughter on his own. Inspired by a true story.


 -- 


Title : Dear Dad
Type : Movie
Director : Tanuj Bhramar
Cast : Arvind Swamy, Himanshu Sharma, Ekavali Khanna, Aman Uppal, Bhavik

### RAG orchestration

In [56]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [59]:
from src.llm import client
from src.llm import generate_answer
print("done")

done


In [60]:
def ask_xpect(query):
    result=retrieve_movies(query)
    context="\n\n -- \n\n".join(result)
    answer=generate_answer(query,context)
    return answer
answer = ask_xpect("movies about a father and daughter")
print(answer)

NotFoundError: Error code: 404 - {'error': {'message': 'The model `gpt-oss-120b` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}